In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.4 GQA / MQA

> 🎯 **Goal:** Shrink the memory attention needs at inference by letting many query
heads share a smaller number of key/value heads.

**Why this matters.** When a model generates text, it caches the keys and values it
has already computed so it does not recompute them for every new token. That store is
the *KV cache*, and at long context lengths it dominates memory and bandwidth, often
more than the weights themselves. Shrinking it is what makes long-context, high-batch
serving affordable, which is why GQA is standard in LLaMA 2/3, Mistral, and friends.

**The intuition.** Standard multi-head attention gives every *head* its own query,
key, and value. The keys and values are the expensive part to cache. The insight: you
can keep all the query heads (queries are where most of the expressivity lives) but
let several of them *share* one set of keys and values. Grouped-query attention (GQA)
groups, say, 4 query heads onto 1 KV head. Multi-query attention (MQA) is the extreme:
every query head shares a single KV head. Fewer KV heads means a proportionally
smaller cache.

**The mechanics.** A *head* is one parallel attention channel. With `n_head` query
heads and `n_kv_head` key/value heads, each KV head is reused by `n_head / n_kv_head`
query heads. The key/value projection (`c_kv`) shrinks in proportion to `n_kv_head`,
and so does the cache it feeds at generation time. Full attention sets
`n_kv_head == n_head`, GQA picks something in between, MQA sets `n_kv_head == 1`. The
query side is untouched, so the model keeps its expressive power while paying far
less for memory.

In [ ]:
from pocketlm import PocketLM, PocketLMConfig

def kv_params(n_kv):
    cfg = PocketLMConfig(vocab_size=20, block_size=16, n_layer=1, n_head=4,
                         n_embd=32, n_kv_head=n_kv)
    attn = PocketLM(cfg).blocks[0].attn
    return sum(p.numel() for p in attn.c_kv.parameters())

print("full (4 KV heads):", kv_params(4))
print("GQA  (2 KV heads):", kv_params(2))
print("MQA  (1 KV head) :", kv_params(1))

**What just happened.** The three printouts trace the KV projection size as you go
from full attention (4 KV heads) to GQA (2) to MQA (1). Each halving of KV heads
halves the `c_kv` projection, and the cache it feeds shrinks by the same factor. The
follow-up forward pass shows an MQA model still produces correctly shaped logits: the
model works, the causal no-future-leak invariant still holds, and the memory bill
drops.

⚠️ **Common pitfalls**
- `n_head` must be divisible by `n_kv_head`, otherwise the query heads cannot be
  grouped evenly onto KV heads.
- MQA's single KV head is the cheapest but can dent quality; GQA is the usual
  sweet spot, recovering most of full attention's quality at a fraction of the cache.
- This saves the *KV cache and the KV projection*, not the query projection or the
  attention compute itself. The win is memory and bandwidth at inference, not raw FLOPs.

🏋️ **Try it yourself.** Add an 8-query-head config and try `n_kv_head` of 8, 4, 2,
and 1. Plot or print `kv_params` against `n_kv_head` and confirm the relationship is
linear, then reason about how much cache memory each choice would save at 4k tokens.

In [ ]:
assert kv_params(1) < kv_params(2) < kv_params(4)
cfg = PocketLMConfig(vocab_size=20, block_size=16, n_layer=1, n_head=4, n_embd=32,
                     n_kv_head=1)
logits, _ = PocketLM(cfg)(torch.randint(0, 20, (1, 8)))
assert logits.shape == (1, 8, 20)